The code is based on the `testing_lsh` notebook we saw during the 3rd tutorial. It has been expanded and adapted to suit the needs of the project.

## 1. Imports & Configuration

In [1]:
import os, glob, re, xml.etree.ElementTree as ET
import json
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix

# ── Dynamic path configuration ────────────────────────────────────────────────
project_root = Path.cwd()
default_data_dir = project_root / "data"

env_data_dir = os.getenv("DATA_DIR")
local_cfg_path = project_root / "paths.local.json"

if env_data_dir:
    data_dir = Path(env_data_dir).expanduser()
elif local_cfg_path.exists():
    data_dir = Path(json.loads(local_cfg_path.read_text())["data_dir"]).expanduser()
else:
    data_dir = default_data_dir

CORPUS_DIR = data_dir / "corpus"
GT_PATH    = data_dir / "ground_truth.tsv"

np.random.seed(42)
print("Imports OK")

debug = False # flag for indicating we're using the alt dataset
print(f"Running in debug mode: {debug}")

Imports OK
Running in debug mode: False


In [2]:
# TODO: remove
if debug:
    from sklearn.datasets import fetch_20newsgroups
    import time

    dataset = fetch_20newsgroups(
        subset='all',
        categories=['comp.graphics', 'sci.med', 'talk.politics.mideast'],
        remove=('headers', 'footers', 'quotes')
    )

    start_time = time.time()

    documents = dataset.data
    print(f"Loaded {len(documents)} documents in {time.time()-start_time:.2f}s")

## 2. Load Ground Truth

In [3]:
gt = pd.read_csv(GT_PATH, sep="\t")
gt.columns = gt.columns.str.strip()

# sort by doc_name to match the order of files in CORPUS_DIR
gt = gt.sort_values("doc_name").reset_index(drop=True)

# remove duplicate doc_name entries if any (keep the first occurrence)
gt = gt.drop_duplicates(subset="doc_name", keep="first").reset_index(drop=True)

# crop doc name to just the the ID after "document" (e.g., "0001" from "suspicious-document0001")
gt["doc_name"] = gt["doc_name"].str.extract(r"document(\d+)")

print(f"Ground truth shape: {gt.shape}")
gt.head(10)


Ground truth shape: (902, 2)


,doc_name,plagiarism
0,00014,1
1,00044,1
2,00052,0
3,00060,0
4,00072,0
5,00083,1
6,00085,1
7,00087,0
8,00088,1
9,00093,1


In [4]:
print(gt["plagiarism"].value_counts().rename({1: "Plagiarised", 0: "Clean"}))

plagiarism
Plagiarised    451
Clean          451
Name: count, dtype: int64


## 3. Parse XML Metadata


In [5]:
def parse_xml_meta(xml_path: Path) -> dict:
    """Return a dict with doc_name, is_plagiarised, and a list of plagiarism features.
    
    The XML reference attribute points to the companion .txt file, e.g.:
        reference="suspicious-document05744.txt"
    Doc names are normalised to the stem without extension.
    """
    tree = ET.parse(xml_path)
    root = tree.getroot()
    doc_ref  = root.get("reference", xml_path.stem + ".txt")
    doc_name = Path(doc_ref).stem   # e.g. "suspicious-document05744"

    features = []
    for feat in root.findall("feature"):
        if feat.get("name") == "artificial-plagiarism":
            features.append({
                "obfuscation"  : feat.get("obfuscation", "none"),
                "this_offset"  : int(feat.get("this_offset", 0)),
                "this_length"  : int(feat.get("this_length", 0)),
                "source_ref"   : feat.get("source_reference", ""),
                "source_offset": int(feat.get("source_offset", 0)),
                "source_length": int(feat.get("source_length", 0)),
            })

    return {"doc_name": doc_name, "is_plagiarised": len(features) > 0, "features": features}

# All XML files are in the single corpus directory
xml_files = sorted(CORPUS_DIR.glob("suspicious-document*.xml"))
print(f"Found {len(xml_files)} XML metadata files")

meta_records = [parse_xml_meta(p) for p in xml_files]
meta_df = pd.DataFrame([{k: v for k, v in r.items() if k != "features"} for r in meta_records])
# change is_plagiarised to match the GT labels (1 for plagiarised, 0 for clean)
meta_df["is_plagiarised"] = meta_df["is_plagiarised"].astype(int)
# crop doc name to just the the ID after "document" (e.g., "0001" from "suspicious-document0001")
meta_df["doc_name"] = meta_df["doc_name"].str.extract(r"document(\d+)")

print(meta_df["is_plagiarised"].value_counts().rename({True: "Plagiarised", False: "Clean"}))
meta_df.head()


Found 901 XML metadata files
is_plagiarised
Clean          451
Plagiarised    450
Name: count, dtype: int64


,doc_name,is_plagiarised
0,00014,1
1,00044,1
2,00052,0
3,00060,0
4,00072,0


# 4. Sanity Check

In [6]:
# Find plagiarised documents present in ground truth but missing from the XML metadata
gt_plag = set(gt.loc[gt["plagiarism"] == 1, "doc_name"])
meta_plag = set(
    meta_df.loc[meta_df["is_plagiarised"] == 1, "doc_name"].str.replace("suspicious-", "", regex=False)
)

missing_from_xml = sorted(gt_plag - meta_plag)
extra_in_xml = sorted(meta_plag - gt_plag)

print(f"Plagiarised docs in GT   : {len(gt_plag)}")
print(f"Plagiarised docs in XML  : {len(meta_plag)}")
print(f"Missing from XML metadata: {len(missing_from_xml)}")
print(f"Extra in XML metadata    : {len(extra_in_xml)}")

if missing_from_xml:
    print("\nMissing document(s):")
    for doc in missing_from_xml:
        print(doc, "-> expected XML:", CORPUS_DIR / f"suspicious-{doc}.xml")

        # drop the missing doc from GT to keep the dataset consistent
        gt = gt[gt["doc_name"] != doc]
        print(f"Dropped {doc} from GT, new GT shape: {gt.shape}")

Plagiarised docs in GT   : 451
Plagiarised docs in XML  : 450
Missing from XML metadata: 1
Extra in XML metadata    : 0

Missing document(s):
06862 -> expected XML: /Users/vitto/Desktop/SP26/DA/project_1/15_Suspicious_Passages/31.Suspicious_Passages/corpus/suspicious-06862.xml
Dropped 06862 from GT, new GT shape: (901, 2)


In [7]:
import xml.etree.ElementTree as ET
from collections import defaultdict

def build_plagiarism_map(xml_dir):
    """
    Build a mapping of document IDs to their plagiarized sources.
    
    Args:
        xml_dir: Path to directory containing suspicious-document*.xml files
        
    Returns:
        Dict[str, List[str]]: Maps document IDs to list of source document IDs they plagiarized from.
                              Documents with 0 plagiarism are not included in the map.
    """
    plagiarism_map = defaultdict(list)
    
    # Get all XML files for suspicious documents
    xml_files = sorted(xml_dir.glob("suspicious-document*.xml"))
    
    for xml_file in xml_files:
        # Extract document ID from filename (e.g., "suspicious-document00925.xml" -> "00925")
        doc_id = xml_file.stem.replace("suspicious-document", "")
        
        try:
            tree = ET.parse(xml_file)
            root = tree.getroot()
            
            # Find all artificial-plagiarism features
            for feature in root.findall(".//feature[@name='artificial-plagiarism']"):
                source_ref = feature.get("source_reference")
                if source_ref:
                    # Extract source document ID from reference
                    # e.g., "source-document02721.txt" -> "02721"
                    source_id = source_ref.replace("source-document", "").replace(".txt", "")
                    plagiarism_map[doc_id].append(source_id)
        except ET.ParseError as e:
            print(f"Error parsing {xml_file}: {e}")
    
    return dict(plagiarism_map)

# Build the plagiarism map
plagiarism_map = build_plagiarism_map(CORPUS_DIR)

In [8]:
# Print summary statistics
total_plagiarized = len(plagiarism_map)
total_docs_in_corpus = len(list(CORPUS_DIR.glob("suspicious-document*.xml")))
total_sources = sum(len(sources) for sources in plagiarism_map.values())

print(f"Total documents in corpus: {total_docs_in_corpus}")
print(f"Documents with plagiarism: {total_plagiarized}")
print(f"Total plagiarism instances: {total_sources}")
print(f"\nExample entries:")
for i, (doc_id, sources) in enumerate(list(plagiarism_map.items())[:5]):
    print(f"  Document {doc_id}: plagiarized from {sources}")

Total documents in corpus: 901
Documents with plagiarism: 450
Total plagiarism instances: 4676

Example entries:
  Document 00014: plagiarized from ['03390', '05760', '04633', '05507', '02701', '00852', '03686', '04633', '04678', '03390', '05822', '06355', '04633', '03686', '03686', '02701', '04633', '03686', '03686', '03686', '03686', '03686', '04633', '05822', '00912']
  Document 00044: plagiarized from ['05976', '05976', '05976', '05976', '05976', '05976']
  Document 00083: plagiarized from ['04661', '04886', '03574', '04886', '04886', '05659', '03574', '02104', '03574', '04023', '03574', '01167', '03574', '03574', '03574', '03574', '04661', '04886', '03574', '01167', '02104']
  Document 00085: plagiarized from ['05039', '05039', '05039']
  Document 00088: plagiarized from ['07074', '05235', '07074', '07074', '05235']


In [9]:
# sanity check: check that the keys in the plagiarism map match the entries with a 1 in the GT
assert set(gt.loc[gt["plagiarism"] == 1, "doc_name"]) == set(plagiarism_map.keys()), "Mismatch between GT and XML metadata for plagiarised documents"
# # if there is a mismatch, print the offending document ID and the expected XML file path
# for doc_id in gt.loc[gt["plagiarism"] == 1, "doc_name"]:
#     if doc_id not in plagiarism_map:
#         print(f"GT indicates {doc_id} is plagiarised but it's missing from XML metadata.")
#         print(f"Expected XML file: {CORPUS_DIR / f'suspicious-{doc_id}.xml'}")

In [10]:
# For each document in the plagiarism map:
# 1) remove source IDs whose source-document*.txt is missing
# 2) if no valid sources remain, remove the document from the map

initial_map_size = len(plagiarism_map)
print(f"Initial plagiarism map size: {initial_map_size}")

removed_sources_total = 0

for doc_id, sources in list(plagiarism_map.items()):
    valid_sources = [
        src_id for src_id in sources
        if (CORPUS_DIR / f"source-document{src_id}.txt").exists()
    ]

    removed_count = len(sources) - len(valid_sources)
    if removed_count > 0:
        removed_sources_total += removed_count
        print(f"Document {doc_id}: removed {removed_count} unavailable source(s).")

    if valid_sources:
        plagiarism_map[doc_id] = valid_sources
    else:
        print(f"Document {doc_id} has no available sources. Removing from map.")
        del plagiarism_map[doc_id]

final_map_size = len(plagiarism_map)
print(f"Final plagiarism map size: {final_map_size}")
print(f"Total unavailable sources removed: {removed_sources_total}")

Initial plagiarism map size: 450
Document 00014: removed 23 unavailable source(s).
Document 00044: removed 6 unavailable source(s).
Document 00044 has no available sources. Removing from map.
Document 00083: removed 21 unavailable source(s).
Document 00083 has no available sources. Removing from map.
Document 00085: removed 3 unavailable source(s).
Document 00085 has no available sources. Removing from map.
Document 00088: removed 5 unavailable source(s).
Document 00088 has no available sources. Removing from map.
Document 00093: removed 3 unavailable source(s).
Document 00093 has no available sources. Removing from map.
Document 00108: removed 65 unavailable source(s).
Document 00108 has no available sources. Removing from map.
Document 00109: removed 65 unavailable source(s).
Document 00117: removed 2 unavailable source(s).
Document 00117 has no available sources. Removing from map.
Document 00123: removed 4 unavailable source(s).
Document 00123 has no available sources. Removing fro

Document 04631: removed 89 unavailable source(s).
Document 04645: removed 26 unavailable source(s).
Document 04688: removed 4 unavailable source(s).
Document 04697: removed 1 unavailable source(s).
Document 04697 has no available sources. Removing from map.
Document 04722: removed 9 unavailable source(s).
Document 04722 has no available sources. Removing from map.
Document 04729: removed 4 unavailable source(s).
Document 04729 has no available sources. Removing from map.
Document 04735: removed 11 unavailable source(s).
Document 04747: removed 6 unavailable source(s).
Document 04747 has no available sources. Removing from map.
Document 04754: removed 4 unavailable source(s).
Document 04754 has no available sources. Removing from map.
Document 04776: removed 23 unavailable source(s).
Document 04789: removed 7 unavailable source(s).
Document 04789 has no available sources. Removing from map.
Document 04800: removed 6 unavailable source(s).
Document 04800 has no available sources. Removin

In [11]:
# list the first five documents in the final plagiarism map as a sanity check
print("\nFirst 5 entries in the final plagiarism map:")
for i, (doc_id, sources) in enumerate(list(plagiarism_map.items())[:10]):
    print(f"  Document {doc_id}: plagiarized from {sources}")


First 5 entries in the final plagiarism map:
  Document 00014: plagiarized from ['00852', '04678']
  Document 00109: plagiarized from ['06229', '05899', '05270', '04973', '06229', '05270', '04973', '04973', '04973', '04973', '05270', '04973']
  Document 00171: plagiarized from ['04040']
  Document 00264: plagiarized from ['01942']
  Document 00357: plagiarized from ['03480', '03480']
  Document 00445: plagiarized from ['01472', '01472']
  Document 00468: plagiarized from ['03711']
  Document 00476: plagiarized from ['03604']
  Document 00511: plagiarized from ['01160', '01160', '01160', '01160']
  Document 00514: plagiarized from ['03649', '03649', '03649']


In [12]:
# size of map (unique keys and unique source IDs across all values)
unique_docs = set(plagiarism_map.keys())
unique_sources = set(src for sources in plagiarism_map.values() for src in sources)
print(f"\nPlagiarism cases with at least one available source: {len(unique_docs)}")
print(f"Plagiarism sources whose .txt is present in corpus: {len(unique_sources)}")


Plagiarism cases with at least one available source: 137
Plagiarism sources whose .txt is present in corpus: 155


## 5. Pre-processing

### 5.1 Load suspicious document texts


In [13]:
def clean_text(text):
    """Clean and normalise text by lowercasing and collapsing whitespace."""
    text = text.strip().lower()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r'[^a-z0-9\s]', '', text) # remove punctuation and special characters
    return text

def load_single_doc(path) -> str:
    text = path.read_text(encoding="utf-8", errors="ignore")
    text = clean_text(text)
    return text

def load_corpus(paths) -> dict:
    """Load a list of .txt Paths → {stem: normalised_text}."""
    corpus = {}
    for p in paths:
        text = load_single_doc(p)
        corpus[p.stem] = text
    return corpus

corpus = sorted(CORPUS_DIR.glob("source-document*.txt"))

# TODO: delete
# selected_docs = sorted(unique_docs.union(unique_sources))
# corpus = [CORPUS_DIR / f"source-document{doc_id}.txt" for doc_id in selected_docs]

if (debug):
    corpus = documents

print(f"Suspicious docs loaded : {len(corpus)}")
# print(f"Suspicious docs loaded : {len(txt_paths)}")


Suspicious docs loaded : 902


In [14]:
# remove the "source-document" prefix and ".txt" suffix to get just the ID for each document in the corpus
if (not debug):
    corpus_ids = [p.stem.replace("source-document", "") for p in corpus]
    print(f"First 5 corpus IDs: {corpus_ids[:5]}")

    # check that the corpus ids match the entries in the ground truth
    # if any of them are missing, report the offending ID and expected file path
    gt_ids = set(gt["doc_name"])
    missing_from_corpus = sorted(gt_ids - set(corpus_ids))
    if missing_from_corpus:
        print(f"\nThe following GT document IDs are missing from the corpus:")
        for doc_id in missing_from_corpus:
            print(f"  {doc_id} -> expected file: {CORPUS_DIR / f'source-document{doc_id}.txt'}")

    # now check the other way around: if there are any corpus IDs that are not in the GT, report them as well
    missing_from_gt = sorted(set(corpus_ids) - gt_ids)
    if missing_from_gt:
        print(f"\nThe following corpus document IDs are missing from the GT:")
        for doc_id in missing_from_gt:
            print(f"  {doc_id} -> file: {CORPUS_DIR / f'source-document{doc_id}.txt'}")

First 5 corpus IDs: ['00014', '00044', '00052', '00060', '00072']

The following corpus document IDs are missing from the GT:
  06862 -> file: /Users/vitto/Desktop/SP26/DA/project_1/15_Suspicious_Passages/31.Suspicious_Passages/corpus/source-document06862.txt


### 5.2 Shingling

In [15]:
K_SHINGLE = 9   # character k-gram size; 9 is typical for long documents

# def build_shingles(text: str, k: int = K_SHINGLE) -> frozenset:
#     """Return the set of all k-character substrings of *text*."""
#     return frozenset(text[i:i+k] for i in range(len(text) - k + 1))

def build_shingles(doc_text: str, k: int):
   # Convert document text into a set of substrings (shingles)
    shingles = []
    for i in range(len(doc_text) - k):
        # shingles.append(doc_text[i:i+k])
        shingle = doc_text[i:i+k]
        # shingle = np.int32(hash(doc_text[i:i+k]))  # hash the shingle to a 32-bit integer
        # shingle = hash(doc_text[i:i+k]) & 0xFFFFFFFF  # hash the shingle to a 32-bit integer
        shingles.append(shingle)
    return set(shingles)

shingles = []
# for name, text in corpus.items():
#     shingles.append(build_shingles(text, K_SHINGLE))

# we load documents one by one and build shingles on the fly to save memory, instead of loading the entire corpus at once

# TODO: Remove
# url = "https://raw.githubusercontent.com/brmson/dataset-sts/master/data/sts/sick2014/SICK_train.txt"

# import requests, io
# text = requests.get(url).text # Sends a GET request to the URL and retrieves the content as plain text

# data = pd.read_csv(io.StringIO(text), sep='\t')
# data.head()

# sentences = data['sentence_A'].tolist()
# for sentence in sentences:
#     shingles.append(build_shingles(sentence, K_SHINGLE))
# print(f"Number of shingle sets: {len(shingles)}")

# end remove

if (debug):
    for doc_text in corpus:
        text = clean_text(doc_text)
        # assert text is not empty after cleaning
        # else, print a warning with the offending document id
        # assert text, f"Warning: document text is empty after cleaning. Original text: {doc_text[:100]}..."
        shingles.append(build_shingles(text, K_SHINGLE))
else:
    for path in corpus:
        text = load_single_doc(path)
        shingles.append(build_shingles(text, K_SHINGLE))
    del text  # free memory

# shingles = {name: build_shingles(text, K_SHINGLE) for name, text in corpus.items()}
print(f"Number of shingle sets: {len(shingles)}")


Number of shingle sets: 902


In [16]:
# print the first 5 shingles of the first document as a sanity check
if (not debug):
    print(f"First 5 shingles of the first document ({corpus[0].stem}):")
    print(list(shingles[0])[:5])

First 5 shingles of the first document (source-document00014):
['794978  8', 'to be at ', 'l and nob', 'rted in a', 'ian cotto']


### 5.3 Build Global Vocabulary & One-Hot Encoding

In [17]:
def build_vocab(shingle_sets: list):
    # Convert list of shingle sets into a global shingle->index map.
    full_set = {item for set_ in shingle_sets for item in set_}
    vocab = {}
    for i, shingle in enumerate(full_set):
        vocab[shingle] = i
    return vocab

def shingle_index_list(shingle_set: set, vocab: dict):
    # Return vocab indices for all shingles present in one document.
    return [vocab[s] for s in shingle_set if s in vocab]

def one_hot(shingles: set, vocab: dict):
    #Convert a set of shingles → numeric vector
    # assert shingle set is not empty
    # assert shingles, "Error: empty shingle set provided to one_hot function."
    if not shingles:
        print(f"Warning: empty shingle set provided to one_hot function. Returning all-zero vector.")
    vec = np.zeros(len(vocab))
    for shingle in shingles:
        idx = vocab[shingle]
        vec[idx] = 1
    # if resulting vector is all zeros, print a warning with the offending shingle set
    # assert np.sum(vec) > 0, f"Error: one-hot vector is all zeros for shingle set: {shingles}"
    if np.sum(vec) == 0:
        print(f"Warning: one-hot vector is all zeros for shingle set: {shingles}")
    # if np.sum(vec) == 0:
        # print(f"Warning: one-hot vector is all zeros for shingle set: {shingles}")
    return vec

In [18]:
# build vocab
vocab = build_vocab(shingles)
print(f"Vocab size: {len(vocab):,}")

# one-hot encode our shingles
# Goal: Convert a set of shingles into a numeric vector (one-hot encoding)
if (debug):
    print("Building dense one-hot matrix (debug mode)...")
    shingles_1hot = []
    for shingle_set in shingles:
        shingles_1hot.append(one_hot(shingle_set, vocab))
    # stack into single numpy array
    shingles_1hot = np.stack(shingles_1hot)
    shingles_1hot.shape

else:
    print("Building sparse one-hot matrix (production mode)...")
    # Build sparse one-hot matrix directly in CSR format.
    rows = []
    cols = []

    for row_idx, shingle_set in enumerate(shingles):
        idxs = shingle_index_list(shingle_set, vocab)
        rows.extend([row_idx] * len(idxs))
        cols.extend(idxs)

    data = np.ones(len(rows), dtype=np.uint8)
    shingles_1hot = csr_matrix(
        (data, (np.array(rows, dtype=np.int32), np.array(cols, dtype=np.int32))),
        shape=(len(shingles), len(vocab)),
        dtype=np.uint8,
    )

    print(f"Sparse one-hot shape: {shingles_1hot.shape}")
    print(f"Non-zeros: {shingles_1hot.nnz:,}")
    print(f"Density: {shingles_1hot.nnz / (shingles_1hot.shape[0] * shingles_1hot.shape[1]):.8f}")

Vocab size: 24,647,636
Building sparse one-hot matrix (production mode)...
Sparse one-hot shape: (902, 24647636)
Non-zeros: 87,778,270
Density: 0.00394826


In [19]:
# all_shingles: set = set()
# for shingle_set in shingles.values():
#     all_shingles.update(shingle_set)

# vocab = {shingle: idx for idx, shingle in enumerate(sorted(all_shingles))}
# VOCAB_SIZE = len(vocab)
# print(f"Vocabulary size: {VOCAB_SIZE:,}")

# def one_hot_vector(shingles: frozenset, vocab: dict) -> np.ndarray:
#     vec = np.zeros(len(vocab), dtype=np.uint8)
#     for s in shingles:
#         if s in vocab:
#             vec[vocab[s]] = 1
#     return vec

# doc_names = sorted(shingles.keys())   # e.g. ["suspicious-document00001", ...]
# print(f"Building one-hot matrix for {len(doc_names)} documents …")
# one_hot_matrix = np.stack([one_hot_vector(shingles[n], vocab) for n in doc_names])
# print(f"One-hot matrix shape: {one_hot_matrix.shape}")


In [20]:
# print the first 10 elements of the vocab to see what the shingles look like
print("First 10 shingles in vocab:")
for i, shingle in enumerate(vocab.keys()):
    if i >= 10:
        break
    print(f"{i}: '{shingle}'")

First 10 shingles in vocab:
0: 'rldp a st'
1: 'emia and '
2: 'nd furbis'
3: 'eer scent'
4: 'ablieau t'
5: 'indad le '
6: 'ile us fo'
7: 'iner in w'
8: 'e so anom'
9: 'ove respo'


In [21]:
# display the first 5 rows of the one-hot matrix as a sanity check
print("First 5 rows of the sparse one-hot matrix:")
# print(shingles_1hot[:5].toarray())
shingles_1hot[:5]

First 5 rows of the sparse one-hot matrix:


<Compressed Sparse Row sparse matrix of dtype 'uint8'
	with 1279059 stored elements and shape (5, 24647636)>

In [22]:
# assert all rows have at least one shingle present (sum > 0)
# row_sums = shingles_1hot.sum(axis=1).A1  # sum across columns, convert to 1D array
row_sums = shingles_1hot.sum(axis=1)  # sum across columns
row_sums.sort()
row_sums

matrix([[ 252723],
        [ 157083],
        [ 169684],
        [ 173711],
        [ 525858],
        [  19500],
        [ 144723],
        [ 518765],
        [  19161],
        [   3338],
        [  25654],
        [  15437],
        [  16880],
        [ 743376],
        [   2740],
        [   8221],
        [   9990],
        [   2738],
        [ 136090],
        [  13430],
        [  10058],
        [ 115112],
        [   3870],
        [  13326],
        [ 134546],
        [   8889],
        [  25979],
        [  22633],
        [ 491704],
        [  26974],
        [ 106327],
        [   3255],
        [   3103],
        [  20816],
        [  11432],
        [ 289902],
        [   9744],
        [ 285188],
        [ 156627],
        [  14081],
        [   8879],
        [   6747],
        [  22672],
        [   2466],
        [ 145280],
        [   2743],
        [   2974],
        [  31173],
        [  16611],
        [   2891],
        [  15714],
        [ 196521],
        [  2

In [23]:
# how many rows sum to 0
num_zero_rows = np.sum(row_sums == 0)
print(f"Number of rows with sum 0: {num_zero_rows}")

Number of rows with sum 0: 0


In [24]:
# assert np.all(row_sums > 0), "All documents should have at least one shingle present (no empty rows)"
if np.any(row_sums == 0):
    print("Warning: Some documents have no shingles present (empty rows in one-hot matrix).")

## 6. MinHash Signatures

In [25]:
def minhash_arr(vocab: dict, resolution: int):
    # Goal: Create a MinHash matrix with random permutations
    # Each row represents a different hash function (permutation of indices)
    length = len(vocab.keys())
    arr = np.zeros((resolution, length))
    for i in range(resolution):
        permutation = np.random.permutation(len(vocab)) + 1
        arr[i, :] = permutation.copy()
    return arr.astype(int)

def get_signature(minhash, vector):
    # Goal: Compute the MinHash signature for a given one-hot vector
    # This produces a compact representation for similarity comparison
    # get index locations of every 1 value in vector
    idx = np.nonzero(vector)[0].tolist()
    # use index locations to pull only +ve positions in minhash
    shingles = minhash[:, idx]
    # find minimum value in each hash vector
    # if shingles is empty (i.e., the document has no shingles), return a signature of all zeros
    if shingles.shape[1] == 0:
        print("Warning: No shingles found for this document. Returning zero signature.")
        return np.zeros(minhash.shape[0], dtype=int)
    signature = np.min(shingles, axis=1) # horizontally
    return signature

In [26]:
n_hashes = 100
arr = minhash_arr(vocab, n_hashes)
# Goal: Generate 100 MinHash functions (random permutations) based on the vocabulary
signatures = []

for vector in shingles_1hot:
    signatures.append(get_signature(arr, vector))
# Goal: Compute a compact MinHash signature for each one-hot encoded sentence

# merge signatures into single array
signatures = np.stack(signatures)
# Goal: Combine all signatures into one matrix (each row = one sentence signature)
signatures.shape

(902, 100)

In [27]:
signatures[0]

array([ 9140314,  1989277,  3021065, 14476772, 15066591, 13568882,
       14772216,  5620857,  2990978,   240718,   999264, 19260682,
       22447916,  2901776, 21702943, 12752685, 12758255, 11897729,
       22972801, 17302843, 12066844, 22516417, 12516026, 14699042,
       11075254, 18597508,  1674318, 11278367, 14990626, 22789763,
        6852533,  9368229, 12542722,  5770345, 14243972, 14031141,
       10223390,  4199555, 10022708, 14997406, 18839908,  6697802,
       23413797, 11615044, 15038722, 11456202, 11540045, 17556771,
        1631407, 17808642, 10299306,  7941555, 13616639, 16309129,
       13838846, 10995160,  9467754, 11468868, 15428618,  3135530,
       10781323, 14860521, 20324056,  3879911,  4046940, 20937978,
       11216839,  9535901, 14461474,  6565955, 20260435,  2830477,
        2955494, 13673061,  4173562,  8631848,  6407427, 24451327,
       23153685, 10130320,  3321536,  8420386, 22772997,  3666121,
       11246347,  8303673, 13021057, 12862062,  5526503,  1289

## 7. Locality-Sensitive Hashing (LSH)


In [28]:
from itertools import combinations

class LSH:
    buckets = []
    counter = 0

    def __init__(self, b):
        self.b = b
        for i in range(b):
            self.buckets.append({})

    def make_subvecs(self, signature):
        l = len(signature)
        assert l % self.b == 0
        r = int(l / self.b)
        # break signature into subvectors
        subvecs = []
        for i in range(0, l, r):
            subvecs.append(signature[i:i+r])
        return np.stack(subvecs)
    
    def add_hash(self, signature):
        subvecs = self.make_subvecs(signature).astype(str)
        for i, subvec in enumerate(subvecs):
            subvec = ','.join(subvec)
            if subvec not in self.buckets[i].keys():
                self.buckets[i][subvec] = []
            self.buckets[i][subvec].append(self.counter)
        self.counter += 1

    def check_candidates(self):
        candidates = []
        for bucket_band in self.buckets:
            keys = bucket_band.keys()
            for bucket in keys:
                hits = bucket_band[bucket]
                if len(hits) > 1:
                    candidates.extend(combinations(hits, 2))
        
        return set(candidates)
        # Goal: Return unique pairs of candidate similar items

In [39]:
b = 20
# Goal: Define the number of bands for LSH

lsh = LSH(b)
# Goal: Initialize the LSH model with b bands (each band will group similar signatures)

for signature in signatures:
    lsh.add_hash(signature)
# Goal: Insert each MinHash signature into LSH buckets to group similar items together

In [40]:
candidate_pairs = lsh.check_candidates()
# Goal: Retrieve all candidate pairs of sentences that were placed in the same bucket
# in at least one band → these are likely similar pairs (filtered by LSH)

len(candidate_pairs)
# Goal: Count how many candidate pairs were found (reduces comparisons vs all possible pairs)

# total possible number of pairs = n choose 2 = n*(n-1)/2
n = len(corpus)
# n = len(txt_paths)
total_pairs = n * (n - 1) // 2
print(f"Total possible pairs: {total_pairs:,}")
print(f"Candidate pairs found: {len(candidate_pairs):,}")

# TODO: too many pairs bro!!!

Total possible pairs: 406,351
Candidate pairs found: 406,351


In [41]:
list(candidate_pairs)[:5]

[(1, 751), (436, 679), (35, 335), (785, 833), (298, 745)]

In [42]:
# list the doc names of the first 5 candidate pairs as a sanity check
if (not debug):
    print("\nFirst 5 candidate pairs (by document name):")
    for i, (idx1, idx2) in enumerate(list(candidate_pairs)[:10]):
        doc1 = corpus[idx1].stem  # e.g. "source-document02721"
        doc1_id = doc1.replace("source-document", "")
        doc2 = corpus[idx2].stem
        doc2_id = doc2.replace("source-document", "")
        # print pair of doc ids and if theyre a known plagiarism case in the GT
        # pair: doc1_id (1|0) - doc2_id (1|0)
        gt1 = gt.loc[gt["doc_name"] == doc1_id, "plagiarism"].values[0]
        gt2 = gt.loc[gt["doc_name"] == doc2_id, "plagiarism"].values[0]
        print(f"  Pair {i+1}: {doc1_id} ({'Plagiarised' if gt1 == 1 else 'Clean'}) - {doc2_id} ({'Plagiarised' if gt2 == 1 else 'Clean'})") 


First 5 candidate pairs (by document name):
  Pair 1: 00044 (Plagiarised) - 06070 (Clean)
  Pair 2: 03670 (Plagiarised) - 05480 (Plagiarised)
  Pair 3: 00317 (Clean) - 02864 (Plagiarised)
  Pair 4: 06287 (Clean) - 06734 (Clean)
  Pair 5: 02511 (Clean) - 05998 (Plagiarised)
  Pair 6: 02064 (Plagiarised) - 04601 (Clean)
  Pair 7: 05212 (Clean) - 07206 (Plagiarised)
  Pair 8: 00447 (Clean) - 04176 (Clean)
  Pair 9: 01099 (Clean) - 02316 (Plagiarised)
  Pair 10: 04841 (Plagiarised) - 07172 (Clean)


# 8. Verification

In [43]:
def jaccard_similarity(set1: set, set2: set) -> float:
    """Compute Jaccard similarity between two sets."""
    intersection = len(set1.intersection(set2))
    union = len(set1.union(set2))
    return intersection / union if union > 0 else 0.0

In [44]:
# THRESHOLD = 0.8  # threshold for considering a pair as truly similar
r = int(n_hashes / b)
THRESHOLD = (1/b) ** (1/r)  # theoretical threshold based on LSH parameters
print(f"Using threshold: {THRESHOLD:.4f} (theoretical for b={b}, r={r}, n_hashes={n_hashes})")

true_positives = []
false_positives = []

total_trials = 1000 # check only the first 1000 candidate pairs for faster testing (TODO: remove?)
i = 0
for idx1, idx2 in list(candidate_pairs):
    if idx1 == idx2:
        # print(f"Skipping self-pair: ({idx1}, {idx2})")
        continue  # skip self-pairs
    if i >= total_trials:
        break
    doc1 = shingles[idx1]
    doc2 = shingles[idx2]
    sim = jaccard_similarity(doc1, doc2)
    
    if sim >= THRESHOLD:  # threshold for considering a pair as truly similar
        true_positives.append((idx1, idx2, sim))
    else:
        false_positives.append((idx1, idx2, sim))

    i += 1

print(f"True positives (similar pairs above threshold): {len(true_positives)}")
print(f"False positives (pairs below threshold): {len(false_positives)}")

Using threshold: 0.5493 (theoretical for b=20, r=5, n_hashes=100)
True positives (similar pairs above threshold): 0
False positives (pairs below threshold): 1000


In [45]:
if (debug):
    num_docs = len(corpus)
    total_possible = num_docs * (num_docs - 1) // 2

    print("\n" + "=" * 60)
    print("FINAL RESULTS")
    print("=" * 60)
    print(f"Dataset: {num_docs} documents from 20 Newsgroups")
    print(f"Total possible comparisons: {total_possible:,}")
    print(f"LSH candidate pairs: {len(candidate_pairs):,}")
    print(f"Speedup: {total_possible / len(candidate_pairs):.1f}x")
    print(f"\nPlagiarism Detection (Jaccard ≥ {THRESHOLD}):")
    print(f"True Positives: {len(true_positives)}")
    print(f"False Positives: {len(false_positives)}")

    # Show an example if we found plagiarism
    if true_positives:
        print("\n" + "=" * 60)
        print("EXAMPLE OF DETECTED PLAGIARISM")
        print("=" * 60)
        doc1, doc2, sim = true_positives[0]
        print(f"Documents {doc1} and {doc2} are {sim*100:.1f}% similar\n")
        print(f"Document {doc1} (first 300 chars):")
        print(documents[doc1][:300] + "...\n")
        print(f"Document {doc2} (first 300 chars):")
        print(documents[doc2][:300] + "...")

In [46]:
# out of all the pairs, list the top 10 with the highest Jaccard similarity as a sanity check
true_positives.sort(key=lambda x: x[2], reverse=True)  # sort by similarity
print("\nTop 10 true positives (by Jaccard similarity):")
for i, (idx1, idx2, sim) in enumerate(true_positives[:10]):
    if idx1 == idx2:
        continue  # skip self-pairs
    doc1 = corpus[idx1].stem
    doc2 = corpus[idx2].stem
    print(f"  Pair {i+1}: {doc1} - {doc2} | Jaccard similarity: {sim:.4f}")


Top 10 true positives (by Jaccard similarity):


In [47]:
# convert the top 10 pairs back into their original document names to see if they are really false positives or if the threshold is just too high (check with the ground truth)
print("\nTop 10 false positives with document names:")
for idx1, idx2, sim in false_positives[:10]:
    doc_name1 = corpus[idx1].stem
    # remove the "source-document" prefix from the doc name to match the GT
    doc_name1 = doc_name1.replace("source-document", "")
    doc_name2 = corpus[idx2].stem
    doc_name2 = doc_name2.replace("source-document", "")
    print(f"Pair ({doc_name1}, {doc_name2}) - Jaccard similarity: {sim:.4f}")

    # check if either document is in the GT as plagiarised
    gt_plag1 = gt.loc[gt["doc_name"] == doc_name1, "plagiarism"].values
    gt_plag2 = gt.loc[gt["doc_name"] == doc_name2, "plagiarism"].values
    if len(gt_plag1) > 0 and gt_plag1[0] == 1:
        print(f"  {doc_name1} is marked as plagiarised in GT")
        # list the docs it plagiarised (check the plagiarism map)
        if doc_name1 in plagiarism_map:
            sources = plagiarism_map[doc_name1]
            print(f"  {doc_name1} plagiarised from: {sources}")
            if doc_name2 in sources:
                print(f"      {doc_name2} is a known source for {doc_name1} in the plagiarism map")
        else:
            print(f"  {doc_name1} has no listed sources in the plagiarism map")
    if len(gt_plag2) > 0 and gt_plag2[0] == 1:
        print(f"  {doc_name2} is marked as plagiarised in GT")
        if doc_name2 in plagiarism_map:
            sources = plagiarism_map[doc_name2]
            print(f"  {doc_name2} plagiarised from: {sources}")
            if doc_name1 in sources:
                print(f"     {doc_name1} is a known source for {doc_name2} in the plagiarism map")
        else:
            print(f"  {doc_name2} has no listed sources in the plagiarism map")

    print()


Top 10 false positives with document names:
Pair (00044, 06070) - Jaccard similarity: 0.0278
  00044 is marked as plagiarised in GT
  00044 has no listed sources in the plagiarism map

Pair (03670, 05480) - Jaccard similarity: 0.0094
  03670 is marked as plagiarised in GT
  03670 plagiarised from: ['02363', '02363']
  05480 is marked as plagiarised in GT
  05480 plagiarised from: ['00787']

Pair (00317, 02864) - Jaccard similarity: 0.0641
  02864 is marked as plagiarised in GT
  02864 plagiarised from: ['00852', '00803', '00803', '00852', '00803', '00803', '00803', '00803', '00803', '00803', '00852', '00858', '00803', '00803', '00852']

Pair (06287, 06734) - Jaccard similarity: 0.0001

Pair (02511, 05998) - Jaccard similarity: 0.0355
  05998 is marked as plagiarised in GT
  05998 has no listed sources in the plagiarism map

Pair (02064, 04601) - Jaccard similarity: 0.0000
  02064 is marked as plagiarised in GT
  02064 plagiarised from: ['04688', '04688', '04688']

Pair (05212, 07206) 

In [48]:
# for the top 5 candidates with the highest Jaccard similarity, load their original text and print which shingles they had in common to see if the similarity is due to a few shared phrases or if they are really very similar across the board
print("\nTop 5 candidate pairs with shared shingles:")
for idx1, idx2 in list(candidate_pairs)[:5]:
    doc_name1 = corpus[idx1].stem.replace("source-document", "")
    doc_name2 = corpus[idx2].stem.replace("source-document", "")
    sim = jaccard_similarity(shingles[idx1], shingles[idx2])
    print(f"Pair ({doc_name1}, {doc_name2}) - Jaccard similarity: {sim:.4f}")

    # load original text for both documents
    text1 = load_single_doc(CORPUS_DIR / f"source-document{doc_name1}.txt")
    text2 = load_single_doc(CORPUS_DIR / f"source-document{doc_name2}.txt")

    # build shingles again to find the common ones
    shingles1 = build_shingles(text1, K_SHINGLE)
    shingles2 = build_shingles(text2, K_SHINGLE)

    common_shingles = shingles1.intersection(shingles2)
    print(f"  Number of shared shingles: {len(common_shingles)}")
    print(f"  Example shared shingles: {list(common_shingles)[:5]}")

     # check if either document is in the GT as plagiarised
    gt_plag1 = gt.loc[gt["doc_name"] == doc_name1, "plagiarism"].values
    gt_plag2 = gt.loc[gt["doc_name"] == doc_name2, "plagiarism"].values
    if len(gt_plag1) > 0 and gt_plag1[0] == 1:
        print(f"  {doc_name1} is marked as plagiarised in GT")
        # list the docs it plagiarised (check the plagiarism map)
        if doc_name1 in plagiarism_map:
            sources = plagiarism_map[doc_name1]
            print(f"  {doc_name1} plagiarised from: {sources}")
    if len(gt_plag2) > 0 and gt_plag2[0] == 1:
        print(f"  {doc_name2} is marked as plagiarised in GT")
        if doc_name2 in plagiarism_map:
            sources = plagiarism_map[doc_name2]
            print(f"  {doc_name2} plagiarised from: {sources}")


Top 5 candidate pairs with shared shingles:
Pair (00044, 06070) - Jaccard similarity: 0.0278
  Number of shared shingles: 19563
  Example shared shingles: ['hen he di', ' by day a', 'ouse has ', 'to treat ', 'on his hi']
  00044 is marked as plagiarised in GT
Pair (03670, 05480) - Jaccard similarity: 0.0094
  Number of shared shingles: 1256
  Example shared shingles: ['that from', ' rise to ', ' love and', 'in thereo', 'n i have ']
  03670 is marked as plagiarised in GT
  03670 plagiarised from: ['02363', '02363']
  05480 is marked as plagiarised in GT
  05480 plagiarised from: ['00787']
Pair (00317, 02864) - Jaccard similarity: 0.0641
  Number of shared shingles: 27950
  Example shared shingles: ['ce howeve', 'it is a s', 'he countr', 'th the co', 'ting the ']
  02864 is marked as plagiarised in GT
  02864 plagiarised from: ['00852', '00803', '00803', '00852', '00803', '00803', '00803', '00803', '00803', '00803', '00852', '00858', '00803', '00803', '00852']
Pair (06287, 06734) - Jacc